Function: remove stationary segments, create fixed windows.

    Windowing strategy:,
    1. Load resampled runs from data/resampled/,
    2. Detect stationary segments using motion metrics (gyro magnitude, acceleration changes),
    3. Remove stationary windows to focus on dynamic motion,
    4. Create fixed-size sliding windows (e.g., 2 seconds at 100 Hz = 200 samples),
    5. Preserve labels and track window-to-label mapping,
    6. Save windowed datasets to data/processed/,


In [4]:
from pathlib import Path
import sys

# Make project root importable whether CWD is repo root or notebooks/
cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / 'src').exists() else cwd.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.io import load_run, discover_run_dirs
from src.windowing import (
    detect_stationary_segments,
    remove_stationary,
    create_windows,
    assign_window_labels,
    save_windowed_run
)



In [5]:
# Load all resampled runs
resampled_data_dir = PROJECT_ROOT / "data" / "resampled"
resampled_run_dirs = sorted(resampled_data_dir.glob("log_*"))

resampled_runs = {}
for run_dir in resampled_run_dirs:
    print(f"Loading {run_dir.name}...")
    try:
        run = load_run(run_dir, include_pose=False)
        resampled_runs[run.run_id] = run
        print(f"  ACC={len(run.acc)}, GYRO={len(run.gyro)}, ODO={len(run.odo)}")
    except Exception as e:
        print(f"  Failed: {e}")

print(f"\nTotal resampled runs: {len(resampled_runs)}")

Loading log_20260223_142511.490...
  ACC=292042, GYRO=292065, ODO=292115
Loading log_20260226_102148.990...
  ACC=217281, GYRO=219335, ODO=218322

Total resampled runs: 2
